# 01 — Data Preparation

Download Binance USDT-M full kline data with all 12 columns.

In [ ]:
import torch, sys
assert torch.cuda.is_available(), "Enable GPU: Runtime → Change runtime → T4"
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_BASE  = '/content/drive/MyDrive/yeniBot'
DATA_DIR    = f'{DRIVE_BASE}/data'
CHECKPT_DIR = f'{DRIVE_BASE}/checkpoints'
os.makedirs(f'{DATA_DIR}/raw', exist_ok=True)
os.makedirs(f'{DATA_DIR}/processed', exist_ok=True)
os.makedirs(CHECKPT_DIR, exist_ok=True)

In [ ]:
import os, sys
REPO_URL = 'https://github.com/umutergul74/yeniBot.git'
REPO_DIR = '/content/yenibot_repo'
if os.path.exists(os.path.join(REPO_DIR, '.git')):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}
sys.path.insert(0, REPO_DIR)
print('After git pull, use Runtime → Restart session before trusting changed imports.')

In [ ]:
!pip install -q -r {REPO_DIR}/requirements.txt

In [ ]:
import yaml
with open(f'{REPO_DIR}/config.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['paths']['data_dir'] = DATA_DIR
cfg['paths']['checkpoint_dir'] = CHECKPT_DIR
cfg

In [ ]:
from yenibot.data import (
    download_full_klines,
    download_funding_rates,
    download_futures_metrics_from_vision,
    validate_full_kline_frame,
)

binance = cfg['binance']
raw_dir = f"{DATA_DIR}/raw"
os.makedirs(raw_dir, exist_ok=True)
downloads = [(binance['primary_interval'], 'btc_1h'), (binance['htf_interval'], 'btc_4h')]
for intrabar_interval in binance.get('intrabar_intervals', []):
    downloads.append((intrabar_interval, f"btc_{intrabar_interval}"))

intrabar_intervals = set(binance.get('intrabar_intervals', []))
for interval, name in downloads:
    df = download_full_klines(
        binance['symbol'],
        interval,
        binance['start_date'],
        binance.get('end_date'),
        base_url=binance['base_url'],
        vision_base_url=binance['vision_base_url'],
        data_source=binance.get('data_source', 'auto'),
        limit=binance['limit'],
        request_sleep_seconds=binance['request_sleep_seconds'],
    )
    gap_multiplier = binance.get('intrabar_max_gap_multiplier', binance['max_gap_multiplier']) if interval in intrabar_intervals else binance['max_gap_multiplier']
    df = validate_full_kline_frame(
        df,
        interval,
        max_gap_multiplier=gap_multiplier,
        zero_volume_policy=binance.get('zero_volume_policy', 'error'),
    )
    dropped = df.attrs.get('dropped_zero_volume_rows', 0)
    if dropped:
        print(f'{name}: dropped {dropped} zero-volume/no-trade rows before validation')
    gap_count = df.attrs.get('gap_count_gt_expected', 0)
    if gap_count:
        print(f'{name}: detected {gap_count} gaps larger than one {interval} bar; max gap {df.attrs.get("max_gap")}; threshold multiplier {gap_multiplier}')
    assert 'taker_buy_base_vol' in df.columns and df['taker_buy_base_vol'].abs().sum() > 0
    out = f'{raw_dir}/{name}.parquet'
    df.to_parquet(out, index=False)
    print(name, len(df), df['timestamp'].min(), df['timestamp'].max(), out)

metrics_cfg = binance.get('futures_metrics', {}) or {}
if metrics_cfg.get('enabled', False):
    metrics = download_futures_metrics_from_vision(
        binance['symbol'],
        binance['start_date'],
        binance.get('end_date'),
        vision_base_url=binance['vision_base_url'],
        request_sleep_seconds=metrics_cfg.get('request_sleep_seconds', 0.0),
    )
    out = f"{raw_dir}/{metrics_cfg.get('filename', 'btc_futures_metrics.parquet')}"
    metrics.to_parquet(out, index=False)
    print('futures_metrics', len(metrics), metrics['timestamp'].min(), metrics['timestamp'].max(), out)

funding_cfg = binance.get('funding_rates', {}) or {}
if funding_cfg.get('enabled', False):
    try:
        funding = download_funding_rates(
            binance['symbol'],
            binance['start_date'],
            binance.get('end_date'),
            base_url=binance['base_url'],
            request_sleep_seconds=funding_cfg.get('request_sleep_seconds', binance['request_sleep_seconds']),
        )
        out = f"{raw_dir}/{funding_cfg.get('filename', 'btc_funding_rates.parquet')}"
        funding.to_parquet(out, index=False)
        print('funding_rates', len(funding), funding['timestamp'].min(), funding['timestamp'].max(), out)
    except Exception as exc:
        print('Funding rate download skipped; Binance REST may be restricted from this runtime:', repr(exc))
